In [ ]:
import pickle

from torch import layout

from adaptive_latents import ArrayWithTime
import pandas
from adaptive_latents.utils import angle_between
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def srs_to_l_df(srs):
    records = []
    for k, sr_list in srs.items():
        for sr_i, sr in enumerate(sr_list):
            latents: ArrayWithTime = sr.log['latents']
            for l_i, l in enumerate(sr.stim_designer.log):
                t_of_stim = l['time_of_stim']
                stim_sample = latents.time_to_sample(t_of_stim)
                old_v = latents[stim_sample-1] - latents[stim_sample-2]
                this_v = latents[stim_sample] - latents[stim_sample-1]
                l['old_v'] = old_v
                l['this_v'] = this_v

                records.append(dict(sr_key=k, sr_i=sr_i, l_i=l_i, l=l))
    return pandas.DataFrame(records)


import sys
sys.path.append("/home/jgould/Documents/2026_paper/code/")
with open("/mnt/data/gould_2026_cache/optim_open_vs_closed_077287296748976.pickle", 'rb') as f:
    data = pickle.load(f)
    df = srs_to_l_df(data)


In [ ]:
df.l[0].keys()

In [ ]:
df['sr_key'].str.split(pat=' ', expand=True)

In [ ]:
# l = df.l[0]
# np.sign(np.linalg.det(np.squeeze([l['v'][:,0], l['s'], l['this_v']])))

In [ ]:
df['theta'] = df['l'].apply(lambda l: angle_between(l['v'], l['observed_s_hat'], radians=False))
df['r'] = df['l'].apply(lambda l: np.linalg.norm(l['observed_s_hat']))

# df['theta'] = df['l'].apply(lambda l: angle_between(l['v'], l['s'], radians=False) )
# df['r'] = df['l'].apply(lambda l: np.linalg.norm(l['s']))


df['t'] = df['l'].apply(lambda l: l['time_of_stim'])

In [ ]:
%matplotlib inline
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, layout='constrained')

min_norm = 10
max_angle = 20

for k, ax in zip(df.sr_key.unique(), axs.flatten()):
    sub_df = df[df.sr_key == k]
    ax.scatter(sub_df['theta'], sub_df['r'], c=sub_df['t'], s=1, label=k)
    patch = plt.Rectangle(xy=(0,min_norm), width=max_angle, height=100, color='r', alpha=.1)
    ax.add_patch(patch)
    ax.set_xlim(xmin=0, xmax=180)
    ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')

    # ax.legend()
    ax.set_ylim(0,60)
    ax.axvline(90, linestyle='--', color='gray', alpha=0.5)

for ax in axs[-1,:]:
    ax.set_xlabel('Angle between v and s (degrees)')

for ax in axs[:,0]:
    ax.set_ylabel('Norm of s')

In [ ]:
%matplotlib inline
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, subplot_kw=dict(projection='polar'), layout='constrained')

for k, ax in zip(df.sr_key.unique(), axs.flatten()):
    sub_df = df[df.sr_key == k]
    ax.scatter(sub_df['theta'] * np.pi/180, sub_df['r'], c=sub_df['t'], s=1, label=k)

    ax.set_ylim(0,40)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    patch = plt.Rectangle(xy=(0,min_norm), width=max_angle * np.pi/180, height=100, color='r', alpha=.1)
    ax.add_patch(patch)


    ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < max_angle) & (sub_df.r > min_norm)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')
